# Data Quality: Yellow Taxi - 8 Dimensiones - NYC TLC

Este notebook evalua la calidad de los datos del Yellow Taxi de NYC TLC
a traves de 8 dimensiones: Completitud, Exactitud, Consistencia, Integridad,
Razonabilidad, Oportunidad, Unicidad y Validez.
Cada dimension produce un puntaje entre 0 y 100% basado en el porcentaje de registros
que superan las verificaciones correspondientes.

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work')
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
from pyspark.sql import functions as F
from src.spark_utils import get_spark
from src.paths import RAW

VEHICLE = 'yellow'
RAW_PATH = str(RAW[VEHICLE])
resultados_dq = {}

In [ ]:
spark = get_spark(app_name=f'TLC_DataQuality_{VEHICLE.upper()}')
df = spark.read.parquet(RAW_PATH).cache()
total = df.count()
print(f'Total registros cargados: {total:,}')


## Dimension 1: Completitud - Tenemos todos los datos necesarios?

In [ ]:
# Dimension 1: Completitud
# Se verifica el porcentaje de valores no nulos en cada columna del esquema
columnas_schema = ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']

# Calcular nulos por columna
exprs_nulos = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in columnas_schema]
nulos_row = df.agg(*exprs_nulos).collect()[0]

completitud_por_columna = {}
for col in columnas_schema:
    nulos = nulos_row[col] if nulos_row[col] is not None else 0
    pct_completo = ((total - nulos) / total) * 100 if total > 0 else 0.0
    completitud_por_columna[col] = round(pct_completo, 2)

# Score = media de completitud entre todas las columnas
score_completitud = round(sum(completitud_por_columna.values()) / len(completitud_por_columna), 2)
registros_con_nulo = df.filter(
    F.greatest(*[F.col(c).isNull().cast("int") for c in columnas_schema]) == 1
).count()

resultados_dq['Completitud'] = {
    'score': score_completitud,
    'descripcion': 'Porcentaje medio de valores no nulos en todas las columnas del esquema',
    'registros_fallidos': registros_con_nulo
}

# Mostrar tabla de completitud por columna
print('--- Dimension 1: Completitud ---')
print(f'{"Columna":<30} {"Completitud (%)":>15}')
print('-' * 47)
for col, pct in completitud_por_columna.items():
    estado = 'OK' if pct >= 95 else ('AVISO' if pct >= 80 else 'CRITICO')
    print(f"{col:<30} {pct:>14.2f}%  [{estado}]")
print('-' * 47)
print(f'Score de Completitud: {score_completitud:.2f}%')
print(f'Registros con al menos un nulo: {registros_con_nulo:,}')

## Dimension 2: Exactitud - Los valores son correctos?

In [ ]:
# Dimension 2: Exactitud
# Los valores de los campos categoricos deben coincidir con los catalogos oficiales de TLC

# Catalogos validos segun TLC para Yellow Taxi
vendor_ids_validos = [1, 2]
payment_types_validos = [1, 2, 3, 4, 5, 6]
ratecode_ids_validos = [1, 2, 3, 4, 5, 6, 99]
store_fwd_validos = ['Y', 'N']

# Un registro es exacto si TODOS sus campos categoricos estan en el catalogo
cond_exactitud = (
    F.col("VendorID").isin(vendor_ids_validos) &
    F.col("payment_type").isin(payment_types_validos) &
    F.col("RatecodeID").isin(ratecode_ids_validos) &
    (
        F.col("store_and_fwd_flag").isin(store_fwd_validos) |
        F.col("store_and_fwd_flag").isNull()
    )
)


# --- OPTIMIZACION ONE-PASS (Auto-generada) ---
_exprs = [
    F.sum(F.when(cond_exactitud, 1).otherwise(0)).alias('registros_exactos'),
    F.sum(F.when(~F.col("VendorID").isin(vendor_ids_validos) | F.col("VendorID").isNull(), 1).otherwise(0)).alias('fallidos_vendor'),
    F.sum(F.when(~F.col("payment_type").isin(payment_types_validos) | F.col("payment_type").isNull(), 1).otherwise(0)).alias('fallidos_payment'),
    F.sum(F.when(~F.col("RatecodeID").isin(ratecode_ids_validos) | F.col("RatecodeID").isNull(), 1).otherwise(0)).alias('fallidos_ratecode'),
    F.sum(F.when(~F.col("store_and_fwd_flag").isin(store_fwd_validos) & F.col("store_and_fwd_flag").isNotNull(), 1).otherwise(0)).alias('fallidos_store')
]
_res = df.agg(*_exprs).collect()[0]
registros_exactos = _res['registros_exactos'] or 0
fallidos_vendor = _res['fallidos_vendor'] or 0
fallidos_payment = _res['fallidos_payment'] or 0
fallidos_ratecode = _res['fallidos_ratecode'] or 0
fallidos_store = _res['fallidos_store'] or 0
# ---------------------------------------------
# registros_exactos (Optimizado en One-Pass)
registros_fallidos_exactitud = total - registros_exactos
score_exactitud = round((registros_exactos / total) * 100, 2) if total > 0 else 0.0

# Desglose por campo
# fallidos_vendor (Optimizado en One-Pass)
# fallidos_payment (Optimizado en One-Pass)
# fallidos_ratecode (Optimizado en One-Pass)
# fallidos_store (Optimizado en One-Pass)

resultados_dq['Exactitud'] = {
    'score': score_exactitud,
    'descripcion': 'Porcentaje de registros con valores categoricos dentro de los catalogos oficiales TLC',
    'registros_fallidos': registros_fallidos_exactitud
}

print('--- Dimension 2: Exactitud ---')
print(f'{"Campo":<25} {"Registros Fuera de Catalogo":>28}')
print('-' * 55)
print(f'{"VendorID":<25} {fallidos_vendor:>28,}')
print(f'{"payment_type":<25} {fallidos_payment:>28,}')
print(f'{"RatecodeID":<25} {fallidos_ratecode:>28,}')
print(f'{"store_and_fwd_flag":<25} {fallidos_store:>28,}')
print('-' * 55)
print(f'Registros fallidos (cualquier campo): {registros_fallidos_exactitud:,}')
print(f'Score de Exactitud: {score_exactitud:.2f}%')

## Dimension 3: Consistencia - Los datos son coherentes entre si?

In [ ]:
# Dimension 3: Consistencia
# Los datos deben ser coherentes entre distintos campos del mismo registro

# Regla 1: fecha de recogida debe ser anterior a la fecha de entrega
cond_tiempo = F.col("tpep_pickup_datetime") < F.col("tpep_dropoff_datetime")

# Regla 2: total_amount debe ser mayor o igual a fare_amount
cond_monto = F.col("total_amount") >= F.col("fare_amount")

# Regla 3: tip_amount debe ser >= 0 cuando el pago es con tarjeta (payment_type == 1)
cond_propina = (
    (F.col("payment_type") == 1) & (F.col("tip_amount") >= 0)
) | (F.col("payment_type") != 1) | F.col("payment_type").isNull()

cond_consistencia = cond_tiempo & cond_monto & cond_propina

# Conteo por regla individual

# --- OPTIMIZACION ONE-PASS (Auto-generada) ---
_exprs = [
    F.sum(F.when(~cond_tiempo | F.col("tpep_pickup_datetime").isNull() | F.col("tpep_dropoff_datetime").isNull(), 1).otherwise(0)).alias('fallidos_tiempo'),
    F.sum(F.when(~cond_monto | F.col("total_amount").isNull() | F.col("fare_amount").isNull(), 1).otherwise(0)).alias('fallidos_monto'),
    F.sum(F.when((F.col("payment_type") == 1) & ((F.col("tip_amount") < 0) | F.col("tip_amount").isNull()), 1).otherwise(0)).alias('fallidos_propina'),
    F.sum(F.when(cond_consistencia, 1).otherwise(0)).alias('registros_consistentes')
]
_res = df.agg(*_exprs).collect()[0]
fallidos_tiempo = _res['fallidos_tiempo'] or 0
fallidos_monto = _res['fallidos_monto'] or 0
fallidos_propina = _res['fallidos_propina'] or 0
registros_consistentes = _res['registros_consistentes'] or 0
# ---------------------------------------------
# fallidos_tiempo (Optimizado en One-Pass)
# fallidos_monto (Optimizado en One-Pass)
# fallidos_propina (Optimizado en One-Pass)

# registros_consistentes (Optimizado en One-Pass)
registros_fallidos_consistencia = total - registros_consistentes
score_consistencia = round((registros_consistentes / total) * 100, 2) if total > 0 else 0.0

resultados_dq['Consistencia'] = {
    'score': score_consistencia,
    'descripcion': 'Porcentaje de registros coherentes: pickup < dropoff, total >= fare, tip >= 0 con tarjeta',
    'registros_fallidos': registros_fallidos_consistencia
}

print('--- Dimension 3: Consistencia ---')
print(f'{"Regla":<45} {"Registros Fallidos":>18}')
print('-' * 65)
print(f'{"pickup < dropoff datetime":<45} {fallidos_tiempo:>18,}')
print(f'{"total_amount >= fare_amount":<45} {fallidos_monto:>18,}')
print(f'{"tip_amount >= 0 con pago tarjeta":<45} {fallidos_propina:>18,}')
print('-' * 65)
print(f'Registros fallidos en alguna regla: {registros_fallidos_consistencia:,}')
print(f'Score de Consistencia: {score_consistencia:.2f}%')

## Dimension 4: Integridad - Se mantienen las relaciones entre tablas?

In [ ]:
# Dimension 4: Integridad Referencial
# Los identificadores deben existir en los catalogos oficiales de TLC

# Catalogo de zonas TLC: LocationID 1-265 (rango oficial)
# Catalogo de RatecodeID: [1,2,3,4,5,6,99]
# Catalogo de VendorID: [1,2]

cond_pu_location = F.col("PULocationID").between(1, 265)
cond_do_location = F.col("DOLocationID").between(1, 265)
cond_ratecode_ref = F.col("RatecodeID").isin([1, 2, 3, 4, 5, 6, 99])
cond_vendor_ref = F.col("VendorID").isin([1, 2])

cond_integridad = cond_pu_location & cond_do_location & cond_ratecode_ref & cond_vendor_ref

# Conteo por campo

# --- OPTIMIZACION ONE-PASS (Auto-generada) ---
_exprs = [
    F.sum(F.when(~cond_pu_location | F.col("PULocationID").isNull(), 1).otherwise(0)).alias('fallidos_pu'),
    F.sum(F.when(~cond_do_location | F.col("DOLocationID").isNull(), 1).otherwise(0)).alias('fallidos_do'),
    F.sum(F.when(~cond_ratecode_ref | F.col("RatecodeID").isNull(), 1).otherwise(0)).alias('fallidos_ratecode_ref'),
    F.sum(F.when(~cond_vendor_ref | F.col("VendorID").isNull(), 1).otherwise(0)).alias('fallidos_vendor_ref'),
    F.sum(F.when(cond_integridad, 1).otherwise(0)).alias('registros_integros')
]
_res = df.agg(*_exprs).collect()[0]
fallidos_pu = _res['fallidos_pu'] or 0
fallidos_do = _res['fallidos_do'] or 0
fallidos_ratecode_ref = _res['fallidos_ratecode_ref'] or 0
fallidos_vendor_ref = _res['fallidos_vendor_ref'] or 0
registros_integros = _res['registros_integros'] or 0
# ---------------------------------------------
# fallidos_pu (Optimizado en One-Pass)
# fallidos_do (Optimizado en One-Pass)
# fallidos_ratecode_ref (Optimizado en One-Pass)
# fallidos_vendor_ref (Optimizado en One-Pass)

# registros_integros (Optimizado en One-Pass)
registros_fallidos_integridad = total - registros_integros
score_integridad = round((registros_integros / total) * 100, 2) if total > 0 else 0.0

resultados_dq['Integridad'] = {
    'score': score_integridad,
    'descripcion': 'Porcentaje de registros con IDs validos en catalogos TLC (zonas 1-265, RatecodeID, VendorID)',
    'registros_fallidos': registros_fallidos_integridad
}

print('--- Dimension 4: Integridad Referencial ---')
print(f'{"Campo":<25} {"Referencia":>20} {"Fallidos":>12}')
print('-' * 59)
print(f'{"PULocationID":<25} {"Zonas TLC 1-265":>20} {fallidos_pu:>12,}')
print(f'{"DOLocationID":<25} {"Zonas TLC 1-265":>20} {fallidos_do:>12,}')
print(f'{"RatecodeID":<25} {"[1,2,3,4,5,6,99]":>20} {fallidos_ratecode_ref:>12,}')
print(f'{"VendorID":<25} {"[1,2]":>20} {fallidos_vendor_ref:>12,}')
print('-' * 59)
print(f'Registros con algun ID fuera de catalogo: {registros_fallidos_integridad:,}')
print(f'Score de Integridad: {score_integridad:.2f}%')

## Dimension 5: Razonabilidad - Los valores estan dentro de rangos esperados?

In [ ]:
# Dimension 5: Razonabilidad
# Los valores numericos deben estar dentro de rangos operacionalmente razonables

# Duracion del viaje en minutos
duracion_min = (
    F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")
) / 60.0

cond_distancia = F.col("trip_distance").between(0, 200)
cond_tarifa = F.col("fare_amount").between(0, 1000)
cond_total = F.col("total_amount").between(0, 1000)
cond_pasajeros = F.col("passenger_count").between(1, 6)
cond_duracion = duracion_min.between(1, 480)
cond_propina_pos = F.col("tip_amount") >= 0

cond_razonabilidad = (
    cond_distancia & cond_tarifa & cond_total &
    cond_pasajeros & cond_duracion & cond_propina_pos
)

# Conteo de fallidos por regla

# --- OPTIMIZACION ONE-PASS (Auto-generada) ---
_exprs = [
    F.sum(F.when(~cond_distancia | F.col("trip_distance").isNull(), 1).otherwise(0)).alias('fallidos_distancia'),
    F.sum(F.when(~cond_tarifa | F.col("fare_amount").isNull(), 1).otherwise(0)).alias('fallidos_tarifa'),
    F.sum(F.when(~cond_total | F.col("total_amount").isNull(), 1).otherwise(0)).alias('fallidos_total_r'),
    F.sum(F.when(~cond_pasajeros | F.col("passenger_count").isNull(), 1).otherwise(0)).alias('fallidos_pasajeros'),
    F.sum(F.when(~duracion_min.between(1, 480) | F.col("tpep_pickup_datetime").isNull() | F.col("tpep_dropoff_datetime").isNull(), 1).otherwise(0)).alias('fallidos_duracion'),
    F.sum(F.when((F.col("tip_amount") < 0) | F.col("tip_amount").isNull(), 1).otherwise(0)).alias('fallidos_propina_r'),
    F.sum(F.when(cond_razonabilidad, 1).otherwise(0)).alias('registros_razonables')
]
_res = df.agg(*_exprs).collect()[0]
fallidos_distancia = _res['fallidos_distancia'] or 0
fallidos_tarifa = _res['fallidos_tarifa'] or 0
fallidos_total_r = _res['fallidos_total_r'] or 0
fallidos_pasajeros = _res['fallidos_pasajeros'] or 0
fallidos_duracion = _res['fallidos_duracion'] or 0
fallidos_propina_r = _res['fallidos_propina_r'] or 0
registros_razonables = _res['registros_razonables'] or 0
# ---------------------------------------------
# fallidos_distancia (Optimizado en One-Pass)
# fallidos_tarifa (Optimizado en One-Pass)
# fallidos_total_r (Optimizado en One-Pass)
# fallidos_pasajeros (Optimizado en One-Pass)
# fallidos_duracion (Optimizado en One-Pass)
# fallidos_propina_r (Optimizado en One-Pass)

# registros_razonables (Optimizado en One-Pass)
registros_fallidos_razonabilidad = total - registros_razonables
score_razonabilidad = round((registros_razonables / total) * 100, 2) if total > 0 else 0.0

resultados_dq['Razonabilidad'] = {
    'score': score_razonabilidad,
    'descripcion': 'Porcentaje de registros con valores numericos dentro de rangos operacionales esperados',
    'registros_fallidos': registros_fallidos_razonabilidad
}

print('--- Dimension 5: Razonabilidad ---')
print(f'{"Regla":<40} {"Rango":>15} {"Fallidos":>12}')
print('-' * 69)
print(f'{"trip_distance":<40} {"[0, 200] millas":>15} {fallidos_distancia:>12,}')
print(f'{"fare_amount":<40} {"[0, 1000] USD":>15} {fallidos_tarifa:>12,}')
print(f'{"total_amount":<40} {"[0, 1000] USD":>15} {fallidos_total_r:>12,}')
print(f'{"passenger_count":<40} {"[1, 6]":>15} {fallidos_pasajeros:>12,}')
print(f'{"duracion del viaje":<40} {"[1, 480] min":>15} {fallidos_duracion:>12,}')
print(f'{"tip_amount":<40} {">= 0":>15} {fallidos_propina_r:>12,}')
print('-' * 69)
print(f'Registros fuera de rango en alguna regla: {registros_fallidos_razonabilidad:,}')
print(f'Score de Razonabilidad: {score_razonabilidad:.2f}%')

## Dimension 6: Oportunidad - Los datos estan actualizados?

In [ ]:
# Dimension 6: Oportunidad
# Los registros deben pertenecer al rango de anios esperado (2019-2025)

anios_validos = [2019, 2020, 2021, 2022, 2023, 2024, 2025]

df_con_anio = df.withColumn("anio_pickup", F.year("tpep_pickup_datetime"))

# Distribucion por anio
dist_anio = (
    df_con_anio
    .groupBy("anio_pickup")
    .count()
    .orderBy("anio_pickup")
    .collect()
)

cond_oportunidad = F.col("anio_pickup").isin(anios_validos)
registros_oportunos = df_con_anio.filter(cond_oportunidad).count()
registros_fallidos_oportunidad = total - registros_oportunos
score_oportunidad = round((registros_oportunos / total) * 100, 2) if total > 0 else 0.0

resultados_dq['Oportunidad'] = {
    'score': score_oportunidad,
    'descripcion': 'Porcentaje de registros con anio de recogida dentro del rango valido 2019-2025',
    'registros_fallidos': registros_fallidos_oportunidad
}

print('--- Dimension 6: Oportunidad ---')
print(f'{"Anio":>6} {"Registros":>15} {"% del Total":>12} {"Estado":>10}')
print('-' * 47)
for row in dist_anio:
    anio = row["anio_pickup"]
    cnt = row["count"]
    pct = (cnt / total * 100) if total > 0 else 0.0
    estado = 'VALIDO' if anio in anios_validos else 'FUERA DE RANGO'
    print(f"{str(anio):>6} {cnt:>15,} {pct:>11.2f}% {estado:>10}")
print('-' * 47)
print(f'Registros fuera de rango: {registros_fallidos_oportunidad:,}')
print(f'Score de Oportunidad: {score_oportunidad:.2f}%')

## Dimension 7: Unicidad - Existen registros duplicados?

In [ ]:
# Dimension 7: Unicidad
# Se detectan registros duplicados basandose en columnas clave del viaje

columnas_clave = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'total_amount'
]

# Contar cuantos grupos tienen mas de un registro (duplicados)
df_grupos = (
    df.groupBy(columnas_clave)
    .count()
    .filter(F.col("count") > 1)
)

# Registros duplicados = suma de todos los conteos en grupos con duplicado
# (se resta 1 por grupo para contar solo los duplicados extra)
df_duplicados_detalle = df_grupos.agg(
    F.sum("count").alias("total_en_grupos_dup"),
    F.count("count").alias("grupos_duplicados")
).collect()[0]

total_en_grupos_dup = df_duplicados_detalle["total_en_grupos_dup"] or 0
grupos_duplicados = df_duplicados_detalle["grupos_duplicados"] or 0
registros_duplicados_extra = total_en_grupos_dup - grupos_duplicados

registros_unicos = total - registros_duplicados_extra
score_unicidad = round((registros_unicos / total) * 100, 2) if total > 0 else 0.0

resultados_dq['Unicidad'] = {
    'score': score_unicidad,
    'descripcion': 'Porcentaje de registros unicos (sin duplicados) basado en columnas clave del viaje',
    'registros_fallidos': int(registros_duplicados_extra)
}

print('--- Dimension 7: Unicidad ---')
print('Columnas clave para deteccion de duplicados:')
for col in columnas_clave:
    print(f"  - {col}")
print()
print(f'Total de registros: {total:,}')
print(f'Grupos con duplicados encontrados: {grupos_duplicados:,}')
print(f'Registros duplicados adicionales (extras): {registros_duplicados_extra:,}')
print(f'Registros unicos: {registros_unicos:,}')
print(f'Score de Unicidad: {score_unicidad:.2f}%')

## Dimension 8: Validez - Los formatos son correctos?

In [ ]:
# Dimension 8: Validez
# Los valores deben cumplir con los formatos y restricciones de dominio esperados

# Regla 1: store_and_fwd_flag debe ser 'Y' o 'N' (no nulo, no otro valor)
cond_flag_valido = F.col("store_and_fwd_flag").isin(["Y", "N"])

# Regla 2: VendorID debe ser entero no nulo
cond_vendor_valido = F.col("VendorID").isNotNull() & F.col("VendorID").cast("int").isNotNull()

# Regla 3: pickup y dropoff datetime no deben ser nulos y anio > 2000
cond_pickup_valido = (
    F.col("tpep_pickup_datetime").isNotNull() &
    (F.year("tpep_pickup_datetime") > 2000)
)
cond_dropoff_valido = (
    F.col("tpep_dropoff_datetime").isNotNull() &
    (F.year("tpep_dropoff_datetime") > 2000)
)

# Regla 4: PULocationID y DOLocationID deben ser enteros positivos
cond_pu_valido = F.col("PULocationID").isNotNull() & (F.col("PULocationID") > 0)
cond_do_valido = F.col("DOLocationID").isNotNull() & (F.col("DOLocationID") > 0)

cond_validez = (
    cond_flag_valido & cond_vendor_valido &
    cond_pickup_valido & cond_dropoff_valido &
    cond_pu_valido & cond_do_valido
)

# Conteo por regla

# --- OPTIMIZACION ONE-PASS (Auto-generada) ---
_exprs = [
    F.sum(F.when(~cond_flag_valido | F.col("store_and_fwd_flag").isNull(), 1).otherwise(0)).alias('fallidos_flag'),
    F.sum(F.when(~cond_vendor_valido, 1).otherwise(0)).alias('fallidos_vendor_v'),
    F.sum(F.when(~cond_pickup_valido, 1).otherwise(0)).alias('fallidos_pickup_v'),
    F.sum(F.when(~cond_dropoff_valido, 1).otherwise(0)).alias('fallidos_dropoff_v'),
    F.sum(F.when(~cond_pu_valido, 1).otherwise(0)).alias('fallidos_pu_v'),
    F.sum(F.when(~cond_do_valido, 1).otherwise(0)).alias('fallidos_do_v'),
    F.sum(F.when(cond_validez, 1).otherwise(0)).alias('registros_validos')
]
_res = df.agg(*_exprs).collect()[0]
fallidos_flag = _res['fallidos_flag'] or 0
fallidos_vendor_v = _res['fallidos_vendor_v'] or 0
fallidos_pickup_v = _res['fallidos_pickup_v'] or 0
fallidos_dropoff_v = _res['fallidos_dropoff_v'] or 0
fallidos_pu_v = _res['fallidos_pu_v'] or 0
fallidos_do_v = _res['fallidos_do_v'] or 0
registros_validos = _res['registros_validos'] or 0
# ---------------------------------------------
# fallidos_flag (Optimizado en One-Pass)
# fallidos_vendor_v (Optimizado en One-Pass)
# fallidos_pickup_v (Optimizado en One-Pass)
# fallidos_dropoff_v (Optimizado en One-Pass)
# fallidos_pu_v (Optimizado en One-Pass)
# fallidos_do_v (Optimizado en One-Pass)

# registros_validos (Optimizado en One-Pass)
registros_fallidos_validez = total - registros_validos
score_validez = round((registros_validos / total) * 100, 2) if total > 0 else 0.0

resultados_dq['Validez'] = {
    'score': score_validez,
    'descripcion': 'Porcentaje de registros con formatos de campo correctos (flag, IDs enteros positivos, fechas > 2000)',
    'registros_fallidos': registros_fallidos_validez
}

print('--- Dimension 8: Validez ---')
print(f'{"Regla de Formato":<45} {"Fallidos":>12}')
print('-' * 59)
print(f'{"store_and_fwd_flag en [Y, N] y no nulo":<45} {fallidos_flag:>12,}')
print(f'{"VendorID entero no nulo":<45} {fallidos_vendor_v:>12,}')
print(f'{"tpep_pickup_datetime no nulo y anio > 2000":<45} {fallidos_pickup_v:>12,}')
print(f'{"tpep_dropoff_datetime no nulo y anio > 2000":<45} {fallidos_dropoff_v:>12,}')
print(f'{"PULocationID entero positivo":<45} {fallidos_pu_v:>12,}')
print(f'{"DOLocationID entero positivo":<45} {fallidos_do_v:>12,}')
print('-' * 59)
print(f'Registros con alguna falla de formato: {registros_fallidos_validez:,}')
print(f'Score de Validez: {score_validez:.2f}%')

## Resumen Final: Puntuacion Global de Calidad de Datos

In [ ]:
# Resumen Final: Consolidacion de las 8 Dimensiones de Calidad

# Orden definido de las dimensiones
orden_dimensiones = [
    'Completitud', 'Exactitud', 'Consistencia', 'Integridad',
    'Razonabilidad', 'Oportunidad', 'Unicidad', 'Validez'
]

# Construir DataFrame de resumen
filas_resumen = []
for dim in orden_dimensiones:
    info = resultados_dq[dim]
    filas_resumen.append({
        'Dimension': dim,
        'Score (%)': info['score'],
        'Registros Fallidos': info['registros_fallidos'],
        'Descripcion': info['descripcion']
    })

df_resumen = pd.DataFrame(filas_resumen)
score_global = round(df_resumen['Score (%)'].mean(), 2)

# Mostrar tabla de resumen
print('=' * 80)
print('RESUMEN DE CALIDAD DE DATOS - YELLOW TAXI')
print('=' * 80)
print(f'{"Dimension":<15} {"Score (%)":>10} {"Reg. Fallidos":>15}')
print('-' * 42)
for _, row in df_resumen.iterrows():
    estado = 'VERDE' if row['Score (%)'] >= 95 else ('NARANJA' if row['Score (%)'] >= 80 else 'ROJO')
    print(f"{row['Dimension']:<15} {row['Score (%)']:.2f}% {row['Registros Fallidos']:>15,}  [{estado}]")
print('-' * 42)
print(f'Score Global de Calidad de Datos: {score_global:.2f}%')
print('=' * 80)

# Definir colores por score
def color_por_score(score):
    if score >= 95:
        return '#2ecc71'
    elif score >= 80:
        return '#f39c12'
    else:
        return '#e74c3c'

colores = [color_por_score(s) for s in df_resumen['Score (%)']]

# Grafico de barras horizontales
fig, ax = plt.subplots(figsize=(12, 6))

barras = ax.barh(
    df_resumen['Dimension'],
    df_resumen['Score (%)'],
    color=colores,
    edgecolor='white',
    height=0.6
)

# Etiquetas de valor sobre cada barra
for barra, score in zip(barras, df_resumen['Score (%)']):
    ax.text(
        min(score + 0.5, 101),
        barra.get_y() + barra.get_height() / 2,
        f"{score:.2f}%",
        va='center', ha='left', fontsize=10, fontweight='bold'
    )

# Lineas de referencia
ax.axvline(x=95, color='#27ae60', linestyle='--', linewidth=1.2, alpha=0.7, label='Umbral Verde (95%)')
ax.axvline(x=80, color='#e67e22', linestyle='--', linewidth=1.2, alpha=0.7, label='Umbral Naranja (80%)')

ax.set_xlim(0, 108)
ax.set_xlabel('Puntuacion (%)', fontsize=12)
ax.set_title(
    f'Puntuacion por Dimension de Calidad (%) - Yellow Taxi\\nScore Global: {score_global:.2f}%',
    fontsize=13, fontweight='bold', pad=15
)

# Leyenda de colores
parche_verde = mpatches.Patch(color='#2ecc71', label='Score >= 95% (Aceptable)')
parche_naranja = mpatches.Patch(color='#f39c12', label='Score >= 80% (Revisar)')
parche_rojo = mpatches.Patch(color='#e74c3c', label='Score < 80% (Critico)')
ax.legend(
    handles=[parche_verde, parche_naranja, parche_rojo],
    loc='lower right', fontsize=10
)

ax.invert_yaxis()
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#ffffff')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('/home/jovyan/work/notebooks/quality/yellow/dq_score_yellow.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Grafico guardado en /home/jovyan/work/notebooks/quality/yellow/dq_score_yellow.png')
print(f'\nScore Global de Calidad de Datos - {title_name}: {{score_global:.2f}}%')

spark.stop()